# In these scenario's we will see how Pydantic is used to validate Agentic-AI data

Scenario 1: The AI Web-Search Tool Call Router

The Scenario: An AI Agent decides it needs to search the web to answer a user's question. Before the agent can call your custom Google Search API tool, Pydantic must validate that the agent generated a properly structured search payload with a safe execution timeout.

In [1]:
from pydantic import BaseModel, Field, field_validator
from typing import Literal

class AgentSearchToolCall(BaseModel):
    # Declaring what engines are allowed
    engine: Literal["google", "bing", "duckduckgo"] = "google" 
    # Defining a search query
    search_query: str = Field(
        description="The clean search string extracted from user intent.",
        min_length=3,
        max_length=100
    )
    timeout_seconds: int = Field(default=5, ge=1, le=15)

    @field_validator('search_query')
    def validate_search_query(cls, value: str) -> str:
        # Removing unnecessary quotes
        clean_query = value.strip().replace('"', '').replace("'","")
        return clean_query

In [4]:
# Mocking an actual tool call api payload following the class structure
ai_payload = {
    "engine": "bing",
    "search_query": " 'Python Pydantic v2 tutorial'",
    "timeout_seconds": 10
}

In [5]:
tool_call = AgentSearchToolCall(**ai_payload)
print(f"🤖 Agent Tool Call Authorized: Routing to {tool_call.engine.upper()} API.")
print(f" Query : {tool_call.search_query} | Timeout : {tool_call.timeout_seconds} s")

🤖 Agent Tool Call Authorized: Routing to BING API.
 Query : Python Pydantic v2 tutorial | Timeout : 10 s


# Example 2: Cross Field Agent Guardrails - Using @model_validator

The Scenario : You have an AI Assistant that helps engineers modify firewall rules via chat. Because AI can sometimes hallucinate (misinterpret context), you need a Pydantic referee to ensure that the agent does not accidentally execute a highly dangerous configuration command like opening port 22 directly to public internet.

In [22]:
from pydantic import BaseModel, Field, model_validator, ValidationError

class FirewallAgentAction(BaseModel):
    server_hostname: str
    target_ip: str
    port: int = Field(ge=1, le=65535)
    action: str = Field(description=" Must be 'ALLOW' or 'DENY'")

    @model_validator(mode='after')
    def enforce_security_check(self) -> 'FirewallAgentAction':
        # Check if target_ip is public internet=0.0.0.0 and port=22 or 80 

        is_public_internet = (self.target_ip == "0.0.0.0" or self.target_ip == "*")
        is_sensitive_port = (self.port in [20,21,22,80])

        if is_public_internet and is_sensitive_port and self.action == "ALLOW":
            raise ValueError(
                f"SECURITY VIOLATION : AI Agent attempted to open sensitive port '{self.port}' over public internet '{self.target_ip}'"
            )

        return self 

In [19]:
# Trying to open a correct port
try:
    port_open1 = FirewallAgentAction(
        server_hostname="webserver1",
        target_ip="10.10.1.10",
        port=8080,
        action="ALLOW"
    )
except ValidationError as ve:
    print("\n🔒 Security Guardrail Tripped! Agent execution forcefully terminated:")
    print(ve.errors()[0]['msg'])

In [20]:
# Validate the request
request_dump = '\n'.join([
    f"Server : {port_open1.server_hostname}",
    f"Target IP : {port_open1.target_ip}",
    f"Port : {port_open1.port}",
    f"Action : {port_open1.action}"

])
print(request_dump)

Server : webserver1
Target IP : 10.10.1.10
Port : 8080
Action : ALLOW


In [23]:
# Trying with an invalid request
# Trying to open a correct port
try:
    port_open2 = FirewallAgentAction(
        server_hostname="webserver1",
        target_ip="0.0.0.0",
        port=80,
        action="ALLOW"
    )
except ValidationError as ve:
    print("\n🔒 Security Guardrail Tripped! Agent execution forcefully terminated:")
    print(ve.errors()[0]['msg'])


🔒 Security Guardrail Tripped! Agent execution forcefully terminated:
Value error, SECURITY VIOLATION : AI Agent attempted to open sensitive port '80' over public internet '0.0.0.0'
